
**论文：** Michael S. Albergo、Nicholas M. Boffi、Eric Vanden-Eijnden，*Stochastic Interpolants: A Unifying Framework for Flows and Diffusions*。

[2023 年 arXiv v3 版本](https://arxiv.org/abs/2303.08797v3) ｜ [论文 HTML 全文](https://arxiv.org/html/2303.08797v3)


# 一、论文到底要解决什么问题？

之前我们学习了三条路线。

**DDPM / Score-SDE：**

先定义一个随机加噪过程：

$$
dX_t=f(X_t,t)\,dt+g(t)\,dW_t
$$

再利用 Score：

$$
s_t(x)=\nabla_x\log\rho_t(x)
$$

构造反向 SDE 或 Probability Flow ODE。

**Flow Matching：**

先规定连接噪声和真实数据的概率路径，例如：

$$
X_t=(1-t)X_0+tX_1
$$

然后回归速度场：

$$
v_\theta(X_t,t)\approx X_1-X_0
$$

**EDM：**

将高斯加噪过程表示为：

$$
X_\sigma=X_{\mathrm{data}}+\sigma Z
$$

再统一设计 Score、去噪器、时间表和求解器。

这三种方法看起来不同，但实际上都在研究同一个问题：

> 如何构造一条从初始概率分布 $\rho_0$ 到目标概率分布 $\rho_1$ 的连续概率路径，并找到能够沿着这条路径生成样本的动力系统？

Stochastic Interpolants 的答案是：

**先定义随机插值，再从随机插值的边缘分布推导 ODE 和 SDE。**

整个理论框架是：

$$
\boxed{
\begin{gathered}
X_t=I(t,X_0,X_1)+\gamma(t)Z\\
\Downarrow\\
\rho_t(x),\quad b_t(x),\quad s_t(x)\\
\Downarrow\\
\partial_t\rho_t=-\nabla\cdot(\rho_tb_t)\\
\Downarrow\\
\begin{cases}
dY_t=b_t(Y_t)\,dt\\
dY_t=(b_t+\varepsilon s_t)\,dt+\sqrt{2\varepsilon}\,dW_t
\end{cases}
\end{gathered}
}
$$

它的重要性不只是统一公式，而是允许我们把以下三种选择分开：

- **概率路径**：如何连接两个分布？
- **学习目标**：学习速度、Score，还是去噪器？
- **采样动力学**：使用确定性 ODE，还是不同随机强度的 SDE？

论文进一步研究：这些选择如何影响似然、数值稳定性和运输成本。[arxiv.org](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 二、第 2.1 节：随机插值的严格定义

## 2.1 随机插值公式

论文定义：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{2.1}
$$

其中：

- $t\in[0,1]$：插值时间。
- $X_0\sim\rho_0$：源分布样本。
- $X_1\sim\rho_1$：目标分布样本。
- $I(t,X_0,X_1)$：插值函数。
- $Z\sim\mathcal N(0,I_d)$：独立标准高斯随机变量。
- $\gamma(t)$：额外噪声的标准差。
- $I_d$：$d$ 维单位矩阵。

注意，$X_0$ 和 $X_1$ **不一定独立**。

论文允许它们来自任意满足指定边缘分布的联合分布：

$$
\boxed{
(X_0,X_1)\sim\nu
}
$$

其中：

$$
\int\nu(dx_0,dx_1)=1
$$

并要求：

$$
\nu(dx_0,\mathbb R^d)=\rho_0(x_0)\,dx_0
$$

$$
\nu(\mathbb R^d,dx_1)=\rho_1(x_1)\,dx_1
$$

这里的 $\nu$ 就是 **Coupling（耦合）**。

因此，你之前研究的 Minibatch OT Coupling，也可以放到这个框架中理解：它改变的是端点的联合采样关系。

## 2.2 端点约束

为了保证：

$$
X_{t=0}\sim\rho_0
$$

$$
X_{t=1}\sim\rho_1
$$

要求：

$$
I(0,X_0,X_1)=X_0
$$

$$
I(1,X_0,X_1)=X_1
$$

以及：

$$
\boxed{
\gamma(0)=\gamma(1)=0
}
$$

所以：

$$
X_{t=0}=X_0
$$

$$
X_{t=1}=X_1
$$

这意味着随机插值在有限时间内精确连接两个端点分布。

与某些需要无限时间才能严格收敛到标准高斯的扩散过程相比，这是一个重要区别。

## 2.3 为什么还要增加 $\gamma(t)Z$？

考虑最简单的线性插值：

$$
I(t,X_0,X_1)=(1-t)X_0+tX_1
$$

不增加噪声时：

$$
X_t=(1-t)X_0+tX_1
$$

增加噪声后：

$$
\boxed{
X_t=(1-t)X_0+tX_1+\gamma(t)Z
}
$$

假设：

$$
\gamma(t)=\sqrt{2t(1-t)}
$$

那么：

$$
\gamma(0)=\gamma(1)=0
$$

中间：

$$
\gamma(0.5)=\frac1{\sqrt2}
$$

所以随机插值会在中间时间额外增加高斯扰动。

为什么这有用？

因为中间边缘分布可以写成：

$$
\boxed{
\rho_t
=
\rho_t^{I}*
\mathcal N(0,\gamma(t)^2I_d)
}
$$

其中 $\rho_t^{I}$ 是不增加 $Z$ 时，插值变量 $I(t,X_0,X_1)$ 的分布。

符号 $*$ 表示卷积。

因此：

$$
\gamma(t)>0
$$

会对中间分布进行高斯平滑。

它能够缓解中间分布过于尖锐、不连续或存在复杂局部结构的问题。

这对速度场学习和数值积分都可能有帮助。

但必须区分：

$$
\boxed{
\gamma(t)\neq\varepsilon(t)
}
$$

其中：

- $\gamma(t)$：改变随机插值本身的中间边缘分布。
- $\varepsilon(t)$：控制后续 SDE 采样时的随机扩散强度。

**前者改变要学习的概率路径，后者可以在保持同一条边缘概率路径的前提下改变采样动力学。**

这是整篇论文最容易混淆、也最重要的概念之一。

---

# 三、第 2.2 节：从随机插值推导连续性方程

现在有：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

我们希望找到一个确定性速度场：

$$
b(t,x)
$$

使得：

$$
\frac{dY_t}{dt}=b(t,Y_t)
$$

能够产生与 $X_t$ 相同的边缘分布。

## 3.1 定义边缘概率密度

随机变量 $X_t$ 的边缘密度：

$$
\boxed{
\rho(t,x)=\mathbb E[\delta(x-X_t)]
}
$$

这里 $\delta$ 是 Dirac delta。

直观上，它把所有可能的随机插值样本在位置 $x$ 处的概率质量累加起来。

接下来计算：

$$
\partial_t\rho(t,x)
$$

## 3.2 对随机插值求时间导数

由于：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

其中 $X_0,X_1,Z$ 在一条插值路径上固定，所以：

$$
\boxed{
\dot X_t
=
\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
}
$$

这里：

$$
\dot\gamma(t)=\frac{d\gamma(t)}{dt}
$$

于是：

$$
\partial_t\rho(t,x)
=
\partial_t\mathbb E[\delta(x-X_t)]
$$

交换微分和期望：

$$
=
\mathbb E[\partial_t\delta(x-X_t)]
$$

根据分布意义下的链式法则：

$$
\partial_t\delta(x-X_t)
=
-\dot X_t^\top\nabla_x\delta(x-X_t)
$$

因此：

$$
\partial_t\rho
=
-\mathbb E[
\dot X_t^\top\nabla_x\delta(x-X_t)
]
$$

将梯度提出期望：

$$
\boxed{
\partial_t\rho
=
-\nabla_x\cdot
\mathbb E[
\dot X_t\delta(x-X_t)
]
}
$$

现在定义：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{2.10}
$$

利用条件期望：

$$
\mathbb E[
\dot X_t\delta(x-X_t)
]
=
\rho(t,x)b(t,x)
$$

所以：

$$
\boxed{
\partial_t\rho(t,x)
+
\nabla_x\cdot[\rho(t,x)b(t,x)]
=0
}
\tag{2.9}
$$

这就是论文的连续性方程。

严格证明需要使用测试函数，避免直接把 Dirac delta 当作普通函数求导；但得到的结果相同。

## 3.3 为什么速度是条件期望？

这个问题非常重要。

假设二维空间中有两条插值路径：

$$
X_t^{(1)}
$$

$$
X_t^{(2)}
$$

它们在某个时刻经过同一个位置：

$$
X_t^{(1)}=X_t^{(2)}=x
$$

但是速度不同：

$$
\dot X_t^{(1)}=(1,0)
$$

$$
\dot X_t^{(2)}=(0,1)
$$

如果希望学习一个确定性速度场：

$$
b(t,x)
$$

那么它不可能同时输出两个不同速度。

对于相同概率权重的两条路径，MSE 最优预测是：

$$
b(t,x)=\left(\frac12,\frac12\right)
$$

这正是：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

所以边缘速度场不一定等于任何一条原始插值路径的速度。

它是所有经过当前状态的条件路径速度的平均。

**这也是为什么 Flow Matching 的训练目标是条件速度，但最终模型学习的是边缘速度场。**

---

# 四、为什么速度场可以通过 MSE 直接训练？

论文定义：

$$
\boxed{
\mathcal L_b[\hat b]
=
\int_0^1
\mathbb E\left[
\frac12\|\hat b(t,X_t)\|^2
-
\dot X_t^\top\hat b(t,X_t)
\right]dt
}
\tag{2.13}
$$

其中：

$$
\dot X_t=\partial_tI+\dot\gamma Z
$$

现在证明它的最优解就是：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

利用条件期望：

$$
\mathbb E[
\dot X_t^\top\hat b(t,X_t)
]
=
\mathbb E[
b(t,X_t)^\top\hat b(t,X_t)
]
$$

所以：

$$
\mathcal L_b
=
\int_0^1
\mathbb E\left[
\frac12\|\hat b\|^2-b^\top\hat b
\right]dt
$$

配方：

$$
\frac12\|\hat b\|^2-b^\top\hat b
=
\frac12\|\hat b-b\|^2-\frac12\|b\|^2
$$

因此：

$$
\boxed{
\mathcal L_b[\hat b]-\mathcal L_b[b]
=
\frac12\int_0^1
\mathbb E\|\hat b(t,X_t)-b(t,X_t)\|^2dt
}
$$

显然：

$$
\boxed{
\hat b^*=b
}
$$

也就是说，实际训练时可以直接使用：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E_{t,X_0,X_1,Z}
\left[
\left\|
b_\theta(X_t,t)
-
\partial_tI
-
\dot\gamma(t)Z
\right\|^2
\right]
}
$$

这个目标不需要：

- 计算 $\rho_t(x)$；
- 计算概率密度的归一化常数；
- 在训练时求解 ODE；
- 在训练时求解 SDE。

只需要能够从端点分布采样，就能构造训练数据。

这就是随机插值框架在生成模型训练中的核心优势。[arXiv](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 五、Score 为什么也能从随机插值直接得到？

论文接下来推导：

$$
\boxed{
s(t,x)=\nabla_x\log\rho(t,x)
}
$$

这里的 Score 是**中间边缘分布的 Score**，不是端点数据分布的 Score。

令：

$$
M_t=I(t,X_0,X_1)
$$

则：

$$
X_t=M_t+\gamma(t)Z
$$

由于：

$$
Z\sim\mathcal N(0,I_d)
$$

给定 $M_t=m$：

$$
X_t\mid M_t=m
\sim
\mathcal N(m,\gamma(t)^2I_d)
$$

所以：

$$
\rho(t,x)
=
\int p_{M_t}(m)
\frac{
\exp\left[-\frac{\|x-m\|^2}{2\gamma^2}\right]
}{
(2\pi\gamma^2)^{d/2}
}
\,dm
$$

其中 $p_{M_t}$ 是插值变量 $M_t$ 的分布。

## 5.1 对密度求梯度

高斯核满足：

$$
\nabla_x
\mathcal N(x;m,\gamma^2I_d)
=
-\frac{x-m}{\gamma^2}
\mathcal N(x;m,\gamma^2I_d)
$$

因此：

$$
\nabla_x\rho(t,x)
=
-\int
\frac{x-m}{\gamma^2}
p_{M_t}(m)
\mathcal N(x;m,\gamma^2I_d)
\,dm
$$

两边除以：

$$
\rho(t,x)
$$

得到：

$$
\nabla_x\log\rho(t,x)
=
-\frac1{\gamma^2}
\mathbb E[X_t-M_t\mid X_t=x]
$$

因为：

$$
X_t-M_t=\gamma Z
$$

所以：

$$
\boxed{
s(t,x)
=
-\frac1{\gamma(t)}
\mathbb E[Z\mid X_t=x]
}
\tag{2.14}
$$

这是论文非常重要的 Score 恒等式。

## 5.2 Score 的训练损失

定义：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

于是：

$$
s(t,x)=-\frac{\eta_z(t,x)}{\gamma(t)}
$$

直接训练 Score：

$$
\boxed{
\mathcal L_s[\hat s]
=
\int_0^1
\mathbb E\left[
\frac12\|\hat s(t,X_t)\|^2
+
\frac{Z^\top\hat s(t,X_t)}{\gamma(t)}
\right]dt
}
\tag{2.16}
$$

它的最优解为：

$$
\hat s^*=s
$$

推导方法与速度场的 MSE 推导完全一致。

但是存在一个实际问题：

$$
\gamma(0)=\gamma(1)=0
$$

所以：

$$
\frac1{\gamma(t)}
$$

在端点附近可能非常大。

这会导致训练目标的数值尺度不稳定。

## 5.3 为什么论文建议预测 $\eta_z$？

既然：

$$
s=-\frac{\eta_z}{\gamma}
$$

我们可以不直接预测 Score，而是训练：

$$
\boxed{
\eta_\theta(t,x)\approx\mathbb E[Z\mid X_t=x]
}
$$

损失：

$$
\boxed{
\mathcal L_{\eta}
=
\mathbb E\|\eta_\theta(t,X_t)-Z\|^2
}
$$

推理时：

$$
\boxed{
s_\theta(t,x)
=
-\frac{\eta_\theta(t,x)}{\gamma(t)}
}
$$

这样训练目标不再包含：

$$
\frac1{\gamma(t)}
$$

可以减轻端点附近的数值问题。

你可以发现，这与 DDPM 中使用噪声预测代替直接预测 Score 的思想高度一致。

但要注意：即使训练 $\eta_z$ 更稳定，推理时如果需要除以非常小的 $\gamma(t)$，仍然需要处理端点数值稳定性。[arXiv](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 六、一个重要分解：速度场 $b$、插值速度 $v$ 与 Score 的关系

论文并不只定义一个速度场。

它还定义：

$$
\boxed{
v(t,x)
=
\mathbb E[
\partial_tI(t,X_0,X_1)
\mid X_t=x
]
}
$$

而完整速度：

$$
b(t,x)
=
\mathbb E[
\partial_tI+\dot\gamma Z
\mid X_t=x
]
$$

根据期望的线性性：

$$
b=v+\dot\gamma\mathbb E[Z\mid X_t=x]
$$

又因为：

$$
\mathbb E[Z\mid X_t=x]=-\gamma s
$$

所以：

$$
\boxed{
b(t,x)
=
v(t,x)-\gamma(t)\dot\gamma(t)s(t,x)
}
\tag{2.27}
$$

这个公式非常关键。

它说明：

$$
\boxed{
\text{边缘速度场}
=
\text{插值函数产生的平均速度}
-
\text{噪声变化引起的 Score 修正}
}
$$

如果：

$$
\gamma(t)=0
$$

并且插值路径的边缘速度场有适当定义，那么：

$$
b=v
$$

就是普通 Flow Matching 的情况。

如果：

$$
\gamma(t)>0
$$

则一般：

$$
b\neq v
$$

这意味着训练时有两种方案。

**方案 A：直接学习 $b$**

监督目标：

$$
\partial_tI+\dot\gamma Z
$$

**方案 B：分别学习 $v$ 和 $s$**

监督：

$$
v_\theta\leftarrow\partial_tI
$$

$$
s_\theta\leftarrow-\frac Z\gamma
$$

然后：

$$
b_\theta=v_\theta-\gamma\dot\gamma s_\theta
$$

论文后面的算法章节会比较这两种方案。

对于你的自动驾驶轨迹规划模型，如果引入额外插值噪声，这个区别就非常重要：**不能仍然直接用 $X_1-X_0$ 作为完整 ODE 速度场的监督目标，而忽略 $\dot\gamma Z$ 或相应的 Score 修正。**

---

# 七、第 2.3 节：如何从同一个随机插值构造一族 SDE？

这是整篇论文最核心的理论结果。

我们已经得到：

$$
\boxed{
\partial_t\rho=-\nabla\cdot(b\rho)
}
$$

现在希望构造：

$$
dY_t=u(t,Y_t)\,dt+\sqrt{2\varepsilon(t)}\,dW_t
$$

其中：

- $u(t,x)$：SDE 漂移。
- $\varepsilon(t)\geq0$：可调扩散系数。
- $W_t$：标准布朗运动。

对应的 Fokker–Planck 方程：

$$
\boxed{
\partial_t\rho
=
-\nabla\cdot(u\rho)+\varepsilon(t)\Delta\rho
}
$$

我们希望它与原来的连续性方程一致。

由于：

$$
s=\nabla\log\rho
$$

有：

$$
\rho s=\nabla\rho
$$

因此：

$$
\Delta\rho=\nabla\cdot(\rho s)
$$

代入：

$$
\partial_t\rho
=
-\nabla\cdot(u\rho)
+
\varepsilon\nabla\cdot(\rho s)
$$

合并：

$$
\partial_t\rho
=
-\nabla\cdot[(u-\varepsilon s)\rho]
$$

为了与：

$$
\partial_t\rho=-\nabla\cdot(b\rho)
$$

一致，可以选择：

$$
u-\varepsilon s=b
$$

因此：

$$
\boxed{
u=b+\varepsilon s
}
$$

最终：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}\,dW_t
}
\tag{2.33}
$$

这就是论文的正向 SDE。

## 7.1 为什么 $\varepsilon$ 可以任意选择？

因为无论 $\varepsilon$ 如何变化：

$$
-\nabla\cdot(\varepsilon s\rho)
+
\varepsilon\Delta\rho=0
$$

也就是说，额外 Score 漂移与随机扩散在 Fokker–Planck 方程中相互抵消。

因此：

$$
\boxed{
\varepsilon(t)\geq0
}
$$

可以自由选择，而不会改变理想情况下的边缘概率路径。

特别地：

当：

$$
\varepsilon=0
$$

得到：

$$
\boxed{
dY_t=b(t,Y_t)\,dt
}
$$

这是确定性 ODE。

当：

$$
\varepsilon>0
$$

得到随机 SDE。

它们的样本轨迹不同，但在精确动力学和相同初始分布下：

$$
\boxed{
\operatorname{Law}(Y_t)=\rho_t
}
$$

这里 $\operatorname{Law}$ 表示随机变量的概率分布。

**注意，这里相同的是每个时刻的边缘分布，不是完整路径的联合分布。**

---

## 7.2 反向 SDE 为什么是 $b-\varepsilon s$？

论文还定义：

$$
\boxed{
b_{\mathrm B}=b-\varepsilon s
}
$$

对应反向 Fokker–Planck 方程：

$$
\boxed{
\partial_t\rho
+
\nabla\cdot(b_{\mathrm B}\rho)
=
-\varepsilon\Delta\rho
}
$$

验证：

$$
\partial_t\rho
=
-\nabla\cdot[(b-\varepsilon s)\rho]
-\varepsilon\Delta\rho
$$

展开：

$$
=
-\nabla\cdot(b\rho)
+
\varepsilon\nabla\cdot(s\rho)
-
\varepsilon\Delta\rho
$$

由于：

$$
\nabla\cdot(s\rho)=\Delta\rho
$$

所以：

$$
\boxed{
\partial_t\rho=-\nabla\cdot(b\rho)
}
$$

依然得到相同的边缘分布。

但反向 SDE 的时间方向需要特别注意。

如果使用：

$$
\tau=1-t
$$

并令反向时间 $\tau$ 从 0 增加到 1，则：

$$
\boxed{
d\widetilde Y_\tau
=
[-b(1-\tau,\widetilde Y_\tau)
+\varepsilon(1-\tau)s(1-\tau,\widetilde Y_\tau)]d\tau
+
\sqrt{2\varepsilon(1-\tau)}\,dW_\tau
}
$$

所以在实际代码中，正向与反向 SDE 不能只通过改变循环顺序来实现，还需要正确处理漂移的时间方向。

---

## 7.3 为什么说它统一了 Score-SDE？

回顾 Score-SDE：

$$
dX_t=f(X_t,t)\,dt+g(t)\,dW_t
$$

对应 Probability Flow ODE：

$$
\frac{dX_t}{dt}
=
f(X_t,t)-\frac12g(t)^2s_t(X_t)
$$

令：

$$
\varepsilon(t)=\frac12g(t)^2
$$

并定义：

$$
b=f-\varepsilon s
$$

那么：

$$
f=b+\varepsilon s
$$

代入：

$$
dX_t=
[b+\varepsilon s]dt+\sqrt{2\varepsilon}\,dW_t
$$

与 Stochastic Interpolants 的正向 SDE 完全一致。

因此：

$$
\boxed{
\text{Score-SDE 的 Probability Flow ODE}
}
$$

可以放进：

$$
\boxed{
\text{Stochastic Interpolants 的连续性方程}
}
$$

框架中理解。

两者主要区别在于：

- Score-SDE 通常先规定 SDE，再推导边缘分布与 ODE。
- Stochastic Interpolants 先规定随机插值的边缘分布，再构造与之匹配的 ODE/SDE。

---

# 八、第 2.4 节：Likelihood Control——为什么 SDE 比 ODE 更容易获得误差保证？

这是上次讲解遗漏最多的部分，也是论文的重要理论贡献。

先定义真实边缘分布：

$$
\rho_t
$$

以及使用学习到的速度场产生的模型分布：

$$
\hat\rho_t
$$

我们希望最终：

$$
\hat\rho_1\approx\rho_1
$$

可以用 KL 散度衡量：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|\hat\rho_1)
=
\int\rho_1(x)
\log\frac{\rho_1(x)}{\hat\rho_1(x)}dx
}
$$

问题是：

**如果我们把速度场的 MSE 训练得很小，是否就能保证最终模型分布的 KL 也很小？**

论文发现，ODE 和 SDE 的情况存在重要区别。

## 8.1 ODE：速度误差不一定足够

假设真实速度：

$$
b
$$

模型速度：

$$
\hat b
$$

对应：

$$
\partial_t\rho=-\nabla\cdot(b\rho)
$$

$$
\partial_t\hat\rho=-\nabla\cdot(\hat b\hat\rho)
$$

并且：

$$
\rho_0=\hat\rho_0
$$

定义：

$$
K(t)=D_{\mathrm{KL}}(\rho_t\|\hat\rho_t)
$$

即：

$$
K(t)=\int\rho\log\frac{\rho}{\hat\rho}dx
$$

对时间求导：

$$
\frac{dK}{dt}
=
\int
\partial_t\rho\log\frac{\rho}{\hat\rho}dx
+
\int\partial_t\rho\,dx
-
\int\rho\frac{\partial_t\hat\rho}{\hat\rho}dx
$$

因为概率密度归一化：

$$
\int\partial_t\rho\,dx=0
$$

所以：

$$
\frac{dK}{dt}
=
\int
\partial_t\rho\log\frac{\rho}{\hat\rho}dx
-
\int\rho\frac{\partial_t\hat\rho}{\hat\rho}dx
$$

代入连续性方程，并进行分部积分。

第一项：

$$
\int
[-\nabla\cdot(b\rho)]
\log\frac{\rho}{\hat\rho}dx
=
\int\rho b^\top
\nabla\log\frac{\rho}{\hat\rho}dx
$$

第二项：

$$
-\int\rho
\frac{-\nabla\cdot(\hat b\hat\rho)}
{\hat\rho}dx
$$

利用：

$$
\frac{\nabla\cdot(\hat b\hat\rho)}{\hat\rho}
=
\nabla\cdot\hat b+
\hat b^\top\nabla\log\hat\rho
$$

整理后得到：

$$
\boxed{
\frac{dK}{dt}
=
\int\rho
(\hat b-b)^\top
(\nabla\log\hat\rho-\nabla\log\rho)
\,dx
}
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|\hat\rho_1)
=
\int_0^1
\mathbb E_{\rho_t}
\left[
(\hat b-b)^\top(\hat s_\rho-s_\rho)
\right]dt
}
\tag{2.39}
$$

这里：

$$
s_\rho=\nabla\log\rho
$$

$$
\hat s_\rho=\nabla\log\hat\rho
$$

注意，$\hat s_\rho$ 是**模型实际生成分布的 Score**，不一定等于单独训练的 Score 网络。

现在使用 Cauchy–Schwarz：

$$
\left|
\frac{dK}{dt}
\right|
\leq
\sqrt{
\mathbb E_\rho\|\hat b-b\|^2
}
\sqrt{
\mathbb E_\rho\|\hat s_\rho-s_\rho\|^2
}
$$

第二项：

$$
\boxed{
\mathcal I(\rho\|\hat\rho)
=
\mathbb E_\rho
\|\nabla\log\rho-\nabla\log\hat\rho\|^2
}
$$

称为 Fisher Divergence。

因此：

$$
\boxed{
\text{仅仅控制速度 MSE，并不能直接控制 ODE 的最终 KL。}
}
$$

因为还涉及真实分布和模型分布的 Score 差异。

这不是说 Flow Matching 的 ODE 不正确，而是说：**仅凭训练速度场的 $L^2$ 误差，不足以得到论文所讨论的这种 KL 上界。**

---

## 8.2 SDE：为什么扩散项能够帮助控制 KL？

现在考虑两个具有相同扩散系数的 SDE。

真实漂移：

$$
u
$$

模型漂移：

$$
\hat u
$$

对应：

$$
\partial_t\rho
=
-\nabla\cdot(u\rho)+\varepsilon\Delta\rho
$$

$$
\partial_t\hat\rho
=
-\nabla\cdot(\hat u\hat\rho)+\varepsilon\Delta\hat\rho
$$

与 ODE 情况类似，对：

$$
K(t)=D_{\mathrm{KL}}(\rho_t\|\hat\rho_t)
$$

求导。

漂移部分仍然产生：

$$
\mathbb E_\rho[
(\hat u-u)^\top(\hat s_\rho-s_\rho)
]
$$

但扩散项会额外产生：

$$
-\varepsilon
\mathbb E_\rho
\|\hat s_\rho-s_\rho\|^2
$$

因此：

$$
\boxed{
\frac{dK}{dt}
=
\mathbb E_\rho[
(\hat u-u)^\top(\hat s_\rho-s_\rho)
]
-
\varepsilon
\mathbb E_\rho
\|\hat s_\rho-s_\rho\|^2
}
$$

这就是论文公式 (2.41) 的核心。

定义：

$$
a=\hat u-u
$$

$$
r=\hat s_\rho-s_\rho
$$

那么：

$$
a^\top r-\varepsilon\|r\|^2
$$

通过配方：

$$
=
-\varepsilon
\left\|
r-\frac{a}{2\varepsilon}
\right\|^2
+
\frac{\|a\|^2}{4\varepsilon}
$$

因为平方项非负：

$$
a^\top r-\varepsilon\|r\|^2
\leq
\frac{\|a\|^2}{4\varepsilon}
$$

所以：

$$
\frac{dK}{dt}
\leq
\frac1{4\varepsilon}
\mathbb E_\rho\|\hat u-u\|^2
$$

积分：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|\hat\rho_1)
\leq
\frac1{4\varepsilon}
\int_0^1
\mathbb E_{\rho_t}
\|\hat u-u\|^2dt
}
\tag{2.42}
$$

这里假设：

$$
\varepsilon>0
$$

且初始分布一致。

**这就是为什么 SDE 可以直接通过漂移的均方误差控制最终 KL。**

因为扩散项提供了一个负的 Fisher Divergence 项。

---

## 8.3 将这个结论用于 Stochastic Interpolants

我们已经知道：

$$
u=b+\varepsilon s
$$

模型使用：

$$
\hat u=\hat b+\varepsilon\hat s
$$

因此：

$$
\hat u-u
=
(\hat b-b)+\varepsilon(\hat s-s)
$$

利用：

$$
\|a+c\|^2
\leq2\|a\|^2+2\|c\|^2
$$

得到：

$$
\|\hat u-u\|^2
\leq
2\|\hat b-b\|^2
+
2\varepsilon^2\|\hat s-s\|^2
$$

代入 KL 上界：

$$
D_{\mathrm{KL}}
\leq
\frac1{2\varepsilon}
\int_0^1\mathbb E\|\hat b-b\|^2dt
+
\frac\varepsilon2
\int_0^1\mathbb E\|\hat s-s\|^2dt
$$

根据前面的二次回归损失：

$$
\mathcal L_b[\hat b]-\mathcal L_b[b]
=
\frac12
\int_0^1\mathbb E\|\hat b-b\|^2dt
$$

Score 也有同样关系。

所以得到：

$$
\boxed{
D_{\mathrm{KL}}(\rho_1\|\hat\rho_1)
\leq
\frac{
\mathcal L_b[\hat b]-\mathcal L_b[b]
}{\varepsilon}
+
\varepsilon
[
\mathcal L_s[\hat s]-\mathcal L_s[s]
]
}
$$

这里按上述损失定义得到这一系数形式；论文中的相应定理采用其自己的规范化写法，阅读时需要留意 $1/2$ 的约定。

**核心结构是：**

$$
\boxed{
D_{\mathrm{KL}}
\lesssim
\frac{\text{速度学习误差}}{\varepsilon}
+
\varepsilon\cdot\text{Score 学习误差}
}
$$

这说明扩散强度并不是越大越好。

如果：

$$
\varepsilon\rightarrow0
$$

速度误差项可能变大。

如果：

$$
\varepsilon\rightarrow\infty
$$

Score 误差项可能变大。

因此存在权衡。

---

## 8.4 为什么论文能够推导最优扩散强度？

令：

$$
A=\mathcal L_b[\hat b]-\min\mathcal L_b
$$

$$
B=\mathcal L_s[\hat s]-\min\mathcal L_s
$$

则上界具有形式：

$$
F(\varepsilon)=\frac A\varepsilon+\varepsilon B
$$

对 $\varepsilon$ 求导：

$$
F'(\varepsilon)
=
-\frac A{\varepsilon^2}+B
$$

令：

$$
F'(\varepsilon)=0
$$

得到：

$$
\frac A{\varepsilon^2}=B
$$

因此：

$$
\boxed{
\varepsilon^*=\sqrt{\frac AB}
}
\tag{2.49}
$$

这意味着：

- 速度场误差较大时，理论上倾向于更大的扩散强度。
- Score 误差较大时，理论上倾向于更小的扩散强度。

但这只是**最小化理论 KL 上界的选择**，不是直接证明生成质量最优。

实际中，较大的扩散强度还可能增加数值积分成本。

这是论文第 2.4 节非常有价值的结论。[arXiv](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 九、第 2.5 节：如何计算 ODE 和 SDE 的似然？

上一节讨论了似然误差的理论上界。

现在的问题是：**给定一个样本 $x$，能不能计算生成模型对它赋予的概率密度？**

## 9.1 ODE 的连续变量替换公式

假设：

$$
\frac{dX_t}{dt}=b(t,X_t)
$$

且：

$$
X_0\sim\rho_0
$$

那么沿着 ODE 轨迹，概率密度满足：

$$
\boxed{
\frac{d}{dt}\log\rho_t(X_t)
=
-\nabla_x\cdot b(t,X_t)
}
$$

我们推导一下。

根据链式法则：

$$
\frac{d}{dt}\log\rho_t(X_t)
=
\partial_t\log\rho_t(X_t)
+
\nabla\log\rho_t(X_t)^\top\dot X_t
$$

代入：

$$
\dot X_t=b(t,X_t)
$$

得到：

$$
=
\frac{\partial_t\rho_t}{\rho_t}
+
b^\top\nabla\log\rho_t
$$

根据连续性方程：

$$
\partial_t\rho_t=-\nabla\cdot(\rho_tb)
$$

展开：

$$
-\nabla\cdot(\rho_tb)
=
-\rho_t\nabla\cdot b
-
b^\top\nabla\rho_t
$$

两边除以 $\rho_t$：

$$
\frac{\partial_t\rho_t}{\rho_t}
=
-\nabla\cdot b-b^\top\nabla\log\rho_t
$$

代回：

$$
\boxed{
\frac{d}{dt}\log\rho_t(X_t)
=
-\nabla\cdot b(t,X_t)
}
$$

积分：

$$
\boxed{
\log\rho_1(X_1)
=
\log\rho_0(X_0)
-
\int_0^1\nabla\cdot b(t,X_t)\,dt
}
$$

因此，只要知道初始分布密度，并且能够求解 ODE 与速度场散度，就可以计算模型的对数似然。

这就是 Continuous Normalizing Flow 的基本公式。

但在高维空间中：

$$
\nabla\cdot b
=
\sum_{j=1}^d
\frac{\partial b_j}{\partial x_j}
$$

可能需要昂贵的 Jacobian 计算，因此通常采用 Hutchinson 迹估计等方法。

## 9.2 SDE 为什么不能直接使用同样的公式？

因为 SDE：

$$
dX_t=u(t,X_t)dt+\sqrt{2\varepsilon}\,dW_t
$$

不是一个普通的确定性可逆映射。

对于固定初始状态，终点仍然是随机变量。

因此不能简单使用 ODE 的 Jacobian 行列式公式。

论文利用 Fokker–Planck 方程和随机过程的期望表示，建立了 SDE 模型的密度计算方法。

其核心是：将密度演化问题转换成沿适当辅助随机过程计算的期望。

这里的具体表达依赖选择正向还是反向 Fokker–Planck 方程，以及辅助过程的初始或终止条件。

**这部分的关键结论是：SDE 也可以进行密度和交叉熵估计，但通常需要随机路径上的期望，而不像 ODE 那样直接沿一条确定性轨迹积分散度。**

---

# 十、第 3 节：不同类型的 Stochastic Interpolants

论文不只研究：

$$
X_t=(1-t)X_0+tX_1+\gamma(t)Z
$$

而是给出了多种特殊结构。

## 10.1 Diffusive Interpolants

第一类是一般的两端点随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

其中：

$$
X_0\sim\rho_0,\qquad X_1\sim\rho_1
$$

两个端点可以是任意分布。

这与传统扩散模型的重要区别是：

$$
\rho_0
$$

不一定必须是标准高斯。

例如可以选择：

- 两个不同的图像域；
- 两种不同的物理状态分布；
- 两种不同的轨迹分布；
- 一个已有生成模型的分布与真实数据分布。

只要能够从两端分布采样，就能构造随机插值训练目标。

## 10.2 One-sided Interpolants：为什么高斯端点可以简化？

现在假设：

$$
\rho_0=\mathcal N(0,I_d)
$$

则：

$$
X_0\sim\mathcal N(0,I_d)
$$

同时：

$$
Z\sim\mathcal N(0,I_d)
$$

如果插值是线性的：

$$
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
$$

那么：

$$
\alpha X_0+\gamma Z
$$

仍然是高斯变量。

因为两个噪声独立：

$$
\operatorname{Var}(\alpha X_0+\gamma Z)
=
(\alpha^2+\gamma^2)I_d
$$

所以可以定义：

$$
\tilde\alpha(t)
=
\sqrt{\alpha(t)^2+\gamma(t)^2}
$$

以及新的标准高斯变量：

$$
\tilde Z_t
=
\frac{\alpha X_0+\gamma Z}
{\sqrt{\alpha^2+\gamma^2}}
$$

对于固定 $t$：

$$
\tilde Z_t\sim\mathcal N(0,I_d)
$$

于是边缘分布可以表示为：

$$
\boxed{
X_t\overset d=
\tilde\alpha(t)\tilde Z+\beta(t)X_1
}
$$

其中 $\overset d=$ 表示同分布。

注意：这里主要是**单时刻边缘分布等价**，不意味着原始过程和重新构造的过程具有相同的跨时间联合分布。

因此，当源分布本来就是高斯时，可以把两份独立高斯噪声合并成一份。

这就是 One-sided Interpolant 能够简化网络设计的原因。

### 10.3 与 VP 扩散过程的关系

考虑：

$$
\boxed{
X_t=\sqrt{1-t^2}\,Z+tX_1
}
$$

其中：

$$
t\in[0,1]
$$

当：

$$
t=0
$$

有：

$$
X_0=Z
$$

当：

$$
t=1
$$

有：

$$
X_1=X_{\mathrm{data}}
$$

这个形式与 VP 扩散模型的边缘分布：

$$
x_\tau=\alpha(\tau)x_{\mathrm{data}}+
\sigma(\tau)\epsilon
$$

非常接近。

只不过这里的时间方向是：

$$
\text{噪声}\rightarrow\text{数据}
$$

而传统前向 VP 扩散是：

$$
\text{数据}\rightarrow\text{噪声}
$$

论文进一步说明，经过适当时间重参数化，Score-Based Diffusion 的边缘路径可以表示为这种 One-sided Interpolant。

## 10.4 Mirror Interpolants

Mirror Interpolant 考虑：

$$
\boxed{
\rho_0=\rho_1
}
$$

也就是说，起始分布和终止分布相同。

例如：

$$
X_t=K(t,X_1)+\gamma(t)Z
$$

其中：

$$
K(0,X_1)=K(1,X_1)=X_1
$$

并且：

$$
\gamma(0)=\gamma(1)=0
$$

所以：

$$
X_0=X_1
$$

$$
X_1=X_1
$$

虽然两端是同一个数据分布，但中间仍然可以经过复杂的概率路径。

例如先逐渐增加噪声，再逐渐去噪。

这类构造的意义在于研究：

**即使起点和终点分布相同，是否仍然可以通过中间的随机过程构造有用的生成和重建机制？**

它也为数据重建、图像变换和中间表示学习提供了不同视角。

---

# 十一、第 3.4 节：Stochastic Interpolants 与 Schrödinger Bridge

这一节是论文理论深度最高的部分之一。

我们先回顾普通最优传输。

## 11.1 最优传输的动力学形式

假设：

$$
\rho_0\rightarrow\rho_1
$$

我们希望寻找一条概率路径：

$$
\rho_t
$$

以及速度场：

$$
v_t(x)
$$

使运输能量最小：

$$
\boxed{
\min_{\rho,v}
\int_0^1
\int
\frac12\|v_t(x)\|^2\rho_t(x)\,dx\,dt
}
$$

约束：

$$
\boxed{
\partial_t\rho_t+\nabla\cdot(\rho_tv_t)=0
}
$$

以及：

$$
\rho_{t=0}=\rho_0
$$

$$
\rho_{t=1}=\rho_1
$$

这就是 Benamou–Brenier 动态最优传输形式。

直观上，它寻找平均动能最小的概率运输路径。

## 11.2 Schrödinger Bridge 增加了扩散

Schrödinger Bridge 的一种流体动力学表示为：

$$
\boxed{
\min_{\rho,u}
\int_0^1
\int
\frac12\|u_t(x)\|^2\rho_t(x)\,dx\,dt
}
$$

约束：

$$
\boxed{
\partial_t\rho_t
+
\nabla\cdot(\rho_tu_t)
=
\varepsilon\Delta\rho_t
}
$$

以及两个端点分布约束。

这里：

$$
\varepsilon>0
$$

表示参考布朗扩散的强度。

与确定性 OT 相比，Schrödinger Bridge 允许概率质量在运输过程中发生随机扩散。

因此它可以看成与熵正则化最优传输密切相关的动态问题。

需要注意，严格的 Schrödinger Bridge 定义还涉及参考路径测度；上述目标采用了相应的动力学表示，并省略了与固定初始分布有关的常数和规范化因素。

## 11.3 为什么最优漂移是梯度场？

我们使用拉格朗日乘子推导。

定义约束：

$$
\partial_t\rho+
\nabla\cdot(\rho u)-\varepsilon\Delta\rho=0
$$

引入乘子：

$$
\lambda(t,x)
$$

拉格朗日泛函：

$$
\begin{aligned}
\mathcal J(\rho,u,\lambda)
={}&
\int_0^1\int
\frac12\|u\|^2\rho\,dx\,dt\\
&+
\int_0^1\int
\lambda[
\partial_t\rho+
\nabla\cdot(\rho u)
-\varepsilon\Delta\rho
]dx\,dt
\end{aligned}
$$

对含 $u$ 的部分做分部积分：

$$
\int\lambda\nabla\cdot(\rho u)dx
=
-\int\rho u^\top\nabla\lambda\,dx
$$

因此与 $u$ 有关的项为：

$$
\int
\rho
\left[
\frac12\|u\|^2-u^\top\nabla\lambda
\right]dx
$$

对 $u$ 求变分：

$$
\frac{\delta\mathcal J}{\delta u}
=
\rho(u-\nabla\lambda)
$$

令其为零：

$$
\boxed{
u^*=\nabla\lambda
}
$$

所以最优漂移是一个梯度场。

## 11.4 推导 Hamilton–Jacobi–Bellman 方程

将：

$$
u=\nabla\lambda
$$

代回泛函。

通过对时间项和扩散项分部积分，得到关于 $\rho$ 的内部积分项：

$$
-\rho\partial_t\lambda
-\frac12\rho\|\nabla\lambda\|^2
-\varepsilon\rho\Delta\lambda
$$

对 $\rho$ 求变分，得到：

$$
\boxed{
\partial_t\lambda
+
\frac12\|\nabla\lambda\|^2
=
-\varepsilon\Delta\lambda
}
$$

因此 Schrödinger Bridge 的最优性条件包括：

$$
\boxed{
\begin{cases}
\partial_t\rho+
\nabla\cdot(\rho\nabla\lambda)
=
\varepsilon\Delta\rho\$$4pt]
\partial_t\lambda+
\frac12\|\nabla\lambda\|^2
=
-\varepsilon\Delta\lambda
\end{cases}
}
$$

第一条描述概率分布如何演化。

第二条描述最优控制势函数如何演化。

这与论文中的 Euler–Lagrange 条件一致。

## 11.5 Stochastic Interpolants 如何求解这个问题？

Stochastic Interpolants 已经构造了一族满足端点约束的概率路径：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

对应：

$$
\rho_t
$$

以及：

$$
u_t=b_t+\varepsilon s_t
$$

因此可以把 Schrödinger Bridge 的优化问题转换为：

**在插值函数 $I$ 的函数空间中优化概率路径，使其对应的随机运输能量最小。**

论文进一步给出了一个关于插值函数和漂移场的 max-min 优化问题：

$$
\boxed{
\max_I\min_{\hat u}
\int_0^1
\mathbb E
\left[
\frac12\|\hat u(t,X_t)\|^2
-
\left(
\partial_tI+
\left[
\dot\gamma-\frac{\varepsilon}{\gamma}
\right]Z
\right)^\top
\hat u(t,X_t)
\right]dt
}
$$

这里的核心是：

$$
\boxed{
u_t=b_t+\varepsilon s_t
}
$$

而：

$$
b_t=
\mathbb E[\partial_tI+\dot\gamma Z\mid X_t]
$$

$$
s_t=
-\frac1\gamma\mathbb E[Z\mid X_t]
$$

因此：

$$
u_t=
\mathbb E
\left[
\partial_tI+
\left(
\dot\gamma-\frac\varepsilon\gamma
\right)Z
\mid X_t
\right]
$$

这正是上面二次目标中监督项的来源。

**需要强调：任意随机插值并不自动是 Schrödinger Bridge。**

只有在相应的优化问题得到正确求解、并满足论文要求的条件时，才可以得到最优桥。

这也是普通 Flow Matching 与 Schrödinger Bridge 的重要区别。[arXiv](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 十二、第 4 节：Spatially Linear Interpolants——线性随机插值的完整推导

这一节与你目前研究的 Flow Matching 最直接相关。

论文考虑：

$$
\boxed{
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
}
\tag{4.1}
$$

其中：

- $\alpha(t)$：源分布样本的权重。
- $\beta(t)$：目标分布样本的权重。
- $\gamma(t)$：额外高斯噪声的权重。

端点约束：

$$
\alpha(0)=1,\quad\alpha(1)=0
$$

$$
\beta(0)=0,\quad\beta(1)=1
$$

$$
\gamma(0)=\gamma(1)=0
$$

## 12.1 为什么速度场可以分解？

首先求导：

$$
\dot X_t
=
\dot\alpha(t)X_0+
\dot\beta(t)X_1+
\dot\gamma(t)Z
$$

定义三个条件期望：

$$
\boxed{
\begin{aligned}
\eta_0(t,x)&=\mathbb E[X_0\mid X_t=x]\\
\eta_1(t,x)&=\mathbb E[X_1\mid X_t=x]\\
\eta_z(t,x)&=\mathbb E[Z\mid X_t=x]
\end{aligned}
}
$$

因此：

$$
b(t,x)
=
\mathbb E[\dot X_t\mid X_t=x]
$$

代入：

$$
\boxed{
b(t,x)
=
\dot\alpha(t)\eta_0(t,x)
+
\dot\beta(t)\eta_1(t,x)
+
\dot\gamma(t)\eta_z(t,x)
}
\tag{4.4}
$$

Score：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\gamma(t)}
}
$$

这说明：**线性随机插值的速度场和 Score 都可以由三个条件期望组合得到。**

但我们是否真的需要训练三个网络？

答案是不一定。

## 12.2 为什么三个条件期望中只需要两个？

因为：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

对条件：

$$
X_t=x
$$

求期望：

$$
\mathbb E[X_t\mid X_t=x]
=
\alpha\mathbb E[X_0\mid X_t=x]
+
\beta\mathbb E[X_1\mid X_t=x]
+
\gamma\mathbb E[Z\mid X_t=x]
$$

左侧：

$$
\mathbb E[X_t\mid X_t=x]=x
$$

因此：

$$
\boxed{
x=\alpha\eta_0+\beta\eta_1+\gamma\eta_z
}
\tag{4.5}
$$

假设：

$$
\alpha(t)\neq0
$$

那么：

$$
\boxed{
\eta_0=
\frac{x-\beta\eta_1-\gamma\eta_z}{\alpha}
}
$$

因此，只要训练：

$$
\eta_1,\quad\eta_z
$$

就可以计算：

$$
\eta_0
$$

进而计算完整速度场。

同样，如果：

$$
\beta(t)\neq0
$$

也可以通过：

$$
\eta_0,\quad\eta_z
$$

计算：

$$
\eta_1
$$

**所以预测空间的选择并不唯一。**

你可以让网络预测：

- 初始噪声；
- 最终数据；
- 插值速度；
- Score；
- 两个端点条件期望的组合。

它们在满足相应条件时，可以通过代数关系相互转换。

但不同预测目标的训练稳定性和数值误差可能不同。

这与你之前研究的 $x_0$ 预测、噪声预测和速度预测的关系直接相关。

---

## 12.3 一个具体例子：线性 CondOT

选择：

$$
\alpha(t)=1-t
$$

$$
\beta(t)=t
$$

$$
\gamma(t)=0
$$

得到：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
$$

对时间求导：

$$
\boxed{
\dot X_t=X_1-X_0
}
$$

所以：

$$
\boxed{
b(t,x)=\eta_1(t,x)-\eta_0(t,x)
}
$$

又因为：

$$
x=(1-t)\eta_0+t\eta_1
$$

可以解出：

$$
\eta_0=\frac{x-t\eta_1}{1-t}
$$

代入：

$$
b=\eta_1-\frac{x-t\eta_1}{1-t}
$$

整理：

$$
\boxed{
b(t,x)=\frac{\eta_1(t,x)-x}{1-t}
}
$$

反过来：

$$
\boxed{
\eta_1(t,x)=x+(1-t)b(t,x)
}
$$

同样：

$$
\boxed{
\eta_0(t,x)=x-tb(t,x)
}
$$

这就是常见 CondOT Flow Matching 中：

$$
\boxed{
\hat x_1=x_t+(1-t)v_\theta(x_t,t)
}
$$

的数学来源。

也就是说：

**速度预测与 $x_1$ 预测在这种线性路径下可以直接相互转换。**

但是，训练速度 MSE 与训练 $x_1$ MSE 的有效时间权重并不相同。

因为：

$$
\hat x_1-x_1
=
(1-t)(v_\theta-v_{\mathrm{target}})
$$

所以：

$$
\boxed{
\|\hat x_1-x_1\|^2
=
(1-t)^2
\|v_\theta-v_{\mathrm{target}}\|^2
}
$$

因此：

- 使用速度 MSE，相当于直接监督速度误差。
- 使用 $x_1$ MSE，相当于对速度误差乘以 $(1-t)^2$。

这解释了为什么预测空间可以等价转换，但不同损失空间的训练效果仍然可能不同。

---

# 十三、不同插值函数会带来什么差异？

论文第 4.2 节比较了多种选择。

### 13.1 Linear Interpolant

$$
\boxed{
\alpha(t)=1-t,\qquad\beta(t)=t
}
$$

特点是条件路径为直线。

如果：

$$
\gamma=0
$$

就是普通线性 Flow Matching。

### 13.2 Trigonometric Interpolant

选择：

$$
\boxed{
\alpha(t)=\cos\frac{\pi t}{2}
}
$$

$$
\boxed{
\beta(t)=\sin\frac{\pi t}{2}
}
$$

如果两个端点独立、均值为零且协方差都是单位矩阵，那么：

$$
\operatorname{Cov}(X_t)
=
\alpha(t)^2I+\beta(t)^2I
$$

由于：

$$
\cos^2\frac{\pi t}{2}
+
\sin^2\frac{\pi t}{2}
=1
$$

因此：

$$
\boxed{
\operatorname{Cov}(X_t)=I
}
$$

这意味着插值过程中总体方差保持不变。

但要注意：**方差保持不变，不代表中间分布的形状保持不变。**

如果端点分布是非高斯的，中间分布依然可能发生复杂变化。

### 13.3 Gaussian Encoding–Decoding

论文还提出一种有意思的插值方式。

让源分布先逐渐退化为高斯噪声，然后从高斯噪声逐渐恢复目标分布。

例如：

$$
\alpha(t)=
\begin{cases}
\cos^2(\pi t),&t<1/2\\
0,&t\geq1/2
\end{cases}
$$

$$
\beta(t)=
\begin{cases}
0,&t\leq1/2\\
\cos^2(\pi t),&t>1/2
\end{cases}
$$

$$
\gamma(t)=\sin^2(\pi t)
$$

当：

$$
t=\frac12
$$

有：

$$
\alpha(1/2)=0
$$

$$
\beta(1/2)=0
$$

$$
\gamma(1/2)=1
$$

因此：

$$
\boxed{
X_{1/2}=Z\sim\mathcal N(0,I)
}
$$

这相当于：

$$
\boxed{
\rho_0
\rightarrow
\mathcal N(0,I)
\rightarrow
\rho_1
}
$$

这种设计的意义是：

**中间经过一个已知的高斯分布，从而避免某些直接插值方式产生的复杂中间模式。**

不过，它可能增加运输路径长度和数值积分成本。

因此并不是所有任务都应该使用这种结构。

---

# 十四、第 4.3 节：$\gamma$ 和 $\varepsilon$ 的区别为什么如此重要？

我们再次比较：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

和：

$$
dY_t=
[b_t(Y_t)+\varepsilon(t)s_t(Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
$$

### 改变 $\gamma$

如果改变：

$$
\gamma(t)
$$

则：

$$
\rho_t
$$

通常会改变。

因此：

$$
b_t,\quad s_t
$$

也会改变。

换句话说：

$$
\boxed{
\gamma\text{ 控制概率路径的设计}
}
$$

### 改变 $\varepsilon$

如果固定：

$$
\rho_t,\quad b_t,\quad s_t
$$

只改变：

$$
\varepsilon(t)
$$

那么在理论上：

$$
\rho_t
$$

保持不变。

但生成样本的路径会发生变化。

因此：

$$
\boxed{
\varepsilon\text{ 控制生成动力学的随机性}
}
$$

这个区别在强化学习中尤其重要。

假设你已经训练好一个条件 Flow Matching 模型：

$$
p_\theta(x_1\mid c)
$$

如果改变训练时的：

$$
\gamma(t)
$$

通常需要重新考虑训练目标和对应的边缘路径。

但如果已经学到准确的：

$$
b_t,\quad s_t
$$

则可以在推理时改变：

$$
\varepsilon(t)
$$

得到不同随机强度的采样过程，而理论目标边缘分布不变。

当然，实际网络存在误差，所以最终分布仍可能随 $\varepsilon$ 改变。

---

# 十五、第 5 节：与其他生成模型的关系

这一节主要讨论三类联系。

## 15.1 与 Score-Based Diffusion 的关系

考虑 Ornstein–Uhlenbeck 过程：

$$
\boxed{
dX_\tau=-X_\tau\,d\tau+\sqrt2\,dW_\tau
}
$$

初始：

$$
X_0\sim\rho_{\mathrm{data}}
$$

这个 SDE 的解析解为：

$$
X_\tau
=
e^{-\tau}X_0+
\sqrt2\int_0^\tau e^{-(\tau-s)}dW_s
$$

由于随机积分是高斯变量，其方差：

$$
2\int_0^\tau e^{-2(\tau-s)}ds
$$

计算：

$$
=2\int_0^\tau e^{-2u}du
$$

$$
=1-e^{-2\tau}
$$

因此：

$$
\boxed{
X_\tau
\overset d=
e^{-\tau}X_0+
\sqrt{1-e^{-2\tau}}Z
}
$$

这就是 VP 扩散模型的一种连续形式。

当：

$$
\tau\rightarrow\infty
$$

有：

$$
e^{-\tau}\rightarrow0
$$

所以：

$$
X_\tau\Rightarrow\mathcal N(0,I)
$$

但对于有限：

$$
\tau=T
$$

通常：

$$
e^{-T}>0
$$

因此：

$$
X_T
$$

并不严格等于标准高斯。

这意味着传统扩散模型在有限时间截断时，可能存在终端分布近似误差。

Stochastic Interpolants 可以直接规定：

$$
X_{t=0}\sim\mathcal N(0,I)
$$

$$
X_{t=1}\sim\rho_{\mathrm{data}}
$$

从而在有限区间内精确满足端点分布约束。

**这并不意味着 Stochastic Interpolants 的数值采样一定更准确，而是说它在理论概率路径的端点定义上避免了这类截断近似。**

---

## 15.2 与 Denoising 的关系

考虑 One-sided Interpolant：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

如果：

$$
\beta(t)\neq0
$$

可以解出：

$$
\boxed{
X_1=\frac{X_t-\alpha(t)Z}{\beta(t)}
}
$$

给定：

$$
X_t=x
$$

取条件期望：

$$
\mathbb E[X_1\mid X_t=x]
=
\frac{
x-\alpha(t)\mathbb E[Z\mid X_t=x]
}{
\beta(t)
}
$$

定义：

$$
\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]
$$

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

得到：

$$
\boxed{
\eta_1(t,x)
=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
}
$$

这个公式与你之前研究的 DDPM、DDIM 中：

$$
\hat x_0=
\frac{x_t-\sigma_t\epsilon_\theta(x_t,t)}{\alpha_t}
$$

具有相同结构。

### 15.3 为什么一次预测不能保证生成正确分布？

虽然：

$$
\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]
$$

是最优 MSE 去噪结果，但：

$$
\eta_1(t,X_t)
$$

的分布通常不等于：

$$
\rho_1
$$

原因是条件期望会平均多个可能的目标样本。

例如，假设：

$$
X_1\in\{-1,1\}
$$

且两个值概率相同。

如果当前状态没有提供任何区分信息，那么：

$$
\mathbb E[X_1\mid X_t]=0
$$

但真实数据分布是：

$$
P(X_1=-1)=P(X_1=1)=\frac12
$$

而不是集中在 0。

因此：

$$
\boxed{
\text{最优 MSE 去噪器}
\neq
\text{直接从目标分布采样的生成器}
}
$$

这也是为什么扩散模型通常需要迭代采样，而不是简单进行一次条件均值预测。

---

## 15.4 迭代 Denoising 为什么会变成 ODE？

论文进一步研究：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

假设已知：

$$
X_t=x
$$

那么对另一时刻 $s$：

$$
X_s=\alpha(s)Z+\beta(s)X_1
$$

取条件期望：

$$
\mathbb E[X_s\mid X_t=x]
=
\alpha(s)\eta_z(t,x)
+
\beta(s)\eta_1(t,x)
$$

代入：

$$
\eta_1(t,x)=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
$$

得到：

$$
\boxed{
\begin{aligned}
\mathbb E[X_s\mid X_t=x]
={}&
\frac{\beta(s)}{\beta(t)}x\\
&+
\left[
\alpha(s)-
\frac{\alpha(t)\beta(s)}{\beta(t)}
\right]\eta_z(t,x)
\end{aligned}
}
\tag{5.13}
$$

现在令：

$$
s=t+h
$$

并对：

$$
\alpha(t+h),\quad\beta(t+h)
$$

进行一阶泰勒展开：

$$
\alpha(t+h)=\alpha(t)+h\dot\alpha(t)+O(h^2)
$$

$$
\beta(t+h)=\beta(t)+h\dot\beta(t)+O(h^2)
$$

因此：

$$
\frac{\beta(t+h)}{\beta(t)}
=
1+h\frac{\dot\beta(t)}{\beta(t)}+O(h^2)
$$

另一项：

$$
\alpha(t+h)-
\frac{\alpha(t)\beta(t+h)}{\beta(t)}
$$

代入：

$$
=
\alpha+h\dot\alpha
-
\alpha\left(
1+h\frac{\dot\beta}{\beta}
\right)
+
O(h^2)
$$

得到：

$$
=
h\left(
\dot\alpha-\alpha\frac{\dot\beta}{\beta}
\right)+O(h^2)
$$

所以：

$$
\begin{aligned}
\mathbb E[X_{t+h}\mid X_t=x]
={}&x\\
&+
h\left[
\frac{\dot\beta}{\beta}x+
\left(
\dot\alpha-\alpha\frac{\dot\beta}{\beta}
\right)\eta_z(t,x)
\right]\\
&+O(h^2)
\end{aligned}
$$

因此，反复进行无穷小步的条件均值更新，对应：

$$
\boxed{
\frac{dX_t}{dt}
=
\frac{\dot\beta(t)}{\beta(t)}X_t
+
\left(
\dot\alpha(t)
-
\alpha(t)\frac{\dot\beta(t)}{\beta(t)}
\right)
\eta_z(t,X_t)
}
$$

这就是对应的 Probability Flow ODE。

**这个推导非常重要：它说明某些迭代去噪公式，在连续时间极限下就是 ODE 数值积分。**

这与我们之前从 DDIM 推导 Probability Flow ODE 的过程是一致的。[arXiv](https://arxiv.org/html/2303.08797v3?utm_source=chatgpt.com)

---

# 十六、第 5.3 节：与 Rectified Flow 的联系

假设我们已经训练了一个速度场：

$$
b(t,x)
$$

并求解：

$$
\frac{dX_t}{dt}=b(t,X_t)
$$

从：

$$
X_0=z
$$

生成：

$$
X_1=T(z)
$$

其中：

$$
T
$$

是 ODE 定义的生成映射。

现在我们可以利用：

$$
(z,T(z))
$$

重新构造一条线性插值：

$$
\boxed{
\widetilde X_t=(1-t)z+tT(z)
}
$$

对应条件速度：

$$
\boxed{
\dot{\widetilde X}_t=T(z)-z
}
$$

再训练新的速度网络：

$$
\widetilde b_\theta
$$

使它拟合：

$$
T(z)-z
$$

这就是 Rectification 的基本思路。

原始生成路径可能比较弯曲，但重新构造的训练路径是连接起点和终点的直线。

通过再次训练，可以尝试获得更容易数值积分的速度场。

不过需要注意：

**条件插值是直线，并不意味着重新学习的边缘 ODE 轨迹严格为直线。**

因为在相同中间状态处，仍然可能存在不同条件速度，需要通过条件期望进行平均。

论文讨论的 Rectification 还强调端点分布的保持，以及近似模型可能引入的偏差问题。

对于你研究的自动驾驶轨迹规划，这一思想可以用于减少 ODE 采样步数，但需要同时考虑训练开销和轨迹质量。

---

# 十七、第 6 节：实际算法如何训练和采样？

现在将前面的理论整理成可以直接实现的形式。

假设选择：

$$
X_t=(1-t)X_0+tX_1+\gamma(t)Z
$$

其中：

$$
X_0\sim\mathcal N(0,I)
$$

$$
X_1\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I)
$$

## 17.1 训练速度场

采样：

$$
t\sim U(0,1)
$$

构造：

$$
X_t=(1-t)X_0+tX_1+\gamma(t)Z
$$

计算：

$$
\dot X_t=X_1-X_0+\dot\gamma(t)Z
$$

训练：

$$
\boxed{
\mathcal L_b
=
\mathbb E
\|b_\theta(X_t,t)-\dot X_t\|^2
}
$$

## 17.2 训练噪声预测器

使用同一个：

$$
X_t
$$

训练：

$$
\boxed{
\mathcal L_\eta
=
\mathbb E
\|\eta_\theta(X_t,t)-Z\|^2
}
$$

然后：

$$
\boxed{
s_\theta(X_t,t)
=
-\frac{\eta_\theta(X_t,t)}{\gamma(t)}
}
$$

注意：这里预测的是**额外插值噪声 $Z$**。

当 $\gamma=0$ 时，不能直接使用这一公式，需要使用其他 Score 参数化或利用高斯端点的特殊结构。

## 17.3 确定性 ODE 采样

从：

$$
X_0\sim\rho_0
$$

开始：

$$
\boxed{
\frac{dX_t}{dt}=b_\theta(X_t,t)
}
$$

使用 Euler：

$$
\boxed{
X_{k+1}
=
X_k+h\,b_\theta(X_k,t_k)
}
$$

或者使用 Heun、RK4 等求解器。

## 17.4 随机 SDE 采样

选择：

$$
\varepsilon(t)>0
$$

构造：

$$
\boxed{
dX_t=
[b_\theta(X_t,t)+\varepsilon(t)s_\theta(X_t,t)]dt
+
\sqrt{2\varepsilon(t)}\,dW_t
}
$$

使用 Euler–Maruyama：

$$
\boxed{
\begin{aligned}
X_{k+1}
={}&X_k+
h[
b_\theta(X_k,t_k)+
\varepsilon(t_k)s_\theta(X_k,t_k)
]\\
&+
\sqrt{2\varepsilon(t_k)h}\,\xi_k
\end{aligned}
}
$$

其中：

$$
\xi_k\sim\mathcal N(0,I)
$$

这是正向时间积分：

$$
t:0\rightarrow1
$$

的公式。

---

# 十八、第 7 节：论文实验验证了什么？

论文主要通过三类实验验证理论。

### 18.1 二维高斯混合分布

使用能够解析计算速度场和 Score 的二维分布，比较：

- 不同插值函数；
- 不同额外噪声 $\gamma$；
- 不同 SDE 扩散强度 $\varepsilon$；
- ODE 与 SDE 的生成轨迹。

这些实验主要说明：

**即使边缘分布相同，不同 $\varepsilon$ 对应的样本路径也可能非常不同。**

同时，增加中间噪声 $\gamma$ 能够平滑概率路径，从而改善某些情况下的数值行为。

### 18.2 高维高斯混合分布

论文还使用高维高斯混合分布研究：

- 学习速度场和 Score 的误差；
- ODE 与 SDE 的分布拟合表现；
- 随机采样对生成质量的影响。

其中一个重要现象是：

**在有限网络容量和有限训练误差下，SDE 采样有时比确定性 ODE 更稳健。**

这与前面第 2.4 节的 KL 误差分析相呼应。

但不能由此得出“SDE 总是比 ODE 好”的结论。

### 18.3 图像生成

论文进一步将方法用于图像生成，研究 One-sided Interpolant、Mirror Interpolant 等设计。

这说明框架不仅适用于低维 toy example，也能够用于高维数据生成。

不过，论文的主要贡献仍然是理论统一与不同设计变量的分析，而不是提出一个在所有基准上都超越已有扩散模型的新网络架构。

---

# 十九、附录 A：为什么高斯混合分布可以解析计算速度和 Score？

这部分值得单独理解，因为它解释了论文为什么大量使用 Gaussian Mixture 作为实验数据。

假设中间分布为：

$$
\boxed{
\rho_t(x)
=
\sum_{k=1}^K
\pi_k\mathcal N(x;\mu_k(t),\Sigma_k(t))
}
$$

其中：

- $K$：高斯分量数量。
- $\pi_k$：混合权重。
- $\mu_k(t)$：第 $k$ 个分量的均值。
- $\Sigma_k(t)$：第 $k$ 个分量的协方差。

定义：

$$
\phi_k(x,t)
=
\mathcal N(x;\mu_k(t),\Sigma_k(t))
$$

则：

$$
\rho_t(x)=\sum_k\pi_k\phi_k(x,t)
$$

对 $x$ 求梯度：

$$
\nabla_x\rho_t(x)
=
\sum_k\pi_k\nabla_x\phi_k(x,t)
$$

高斯分布满足：

$$
\nabla_x\phi_k
=
-\phi_k\Sigma_k^{-1}(x-\mu_k)
$$

所以：

$$
\nabla_x\rho_t
=
-\sum_k
\pi_k\phi_k
\Sigma_k^{-1}(x-\mu_k)
$$

两边除以 $\rho_t$：

$$
\boxed{
s_t(x)
=
-\sum_k
r_k(x,t)
\Sigma_k(t)^{-1}(x-\mu_k(t))
}
$$

其中：

$$
\boxed{
r_k(x,t)
=
\frac{
\pi_k\phi_k(x,t)
}{
\sum_j\pi_j\phi_j(x,t)
}
}
$$

这就是 GMM 的后验责任概率。

你之前研究 MoE 和 GMM 时接触过同样的形式。

因此，对于高斯混合分布：

$$
s_t(x)
$$

可以解析计算。

同样，如果知道每个分量对应的插值动力学，也可以计算其条件速度场，再通过后验责任概率进行加权。

这让作者能够将数值积分误差与网络学习误差分开分析。

---

# 二十、附录 B：主要理论证明之间的联系

论文附录 B 的证明主要围绕以下逻辑展开。

首先，通过随机插值的定义：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

证明其边缘密度满足适当的正则性条件，并且：

$$
\partial_t\rho+\nabla\cdot(b\rho)=0
$$

其次，通过条件期望的 $L^2$ 投影性质，证明：

$$
b=\mathbb E[\dot X_t\mid X_t=x]
$$

以及：

$$
s=-\gamma^{-1}\mathbb E[Z\mid X_t=x]
$$

分别是相应二次损失的最优解。

随后利用：

$$
\rho s=\nabla\rho
$$

证明正向和反向 Fokker–Planck 方程。

接着通过对 KL 散度求时间导数，得到 ODE 和 SDE 在误差控制方面的差异。

最后，使用随机过程与最优控制理论，建立似然估计、Schrödinger Bridge 和其他特殊插值构造的结果。

因此，附录中的数学证明并不是彼此孤立的。

它们围绕同一条主线：

$$
\boxed{
\text{随机插值}
\rightarrow
\text{条件期望}
\rightarrow
\text{概率演化方程}
\rightarrow
\text{生成动力学}
\rightarrow
\text{误差与最优性分析}
}
$$

---

# 二十一、结合你的自动驾驶 Diffusion Planner，最重要的研究启发

最后把论文与我们之前讨论过的轨迹规划问题联系起来。

假设你的轨迹规划模型使用：

$$
X_t=(1-t)X_0+tX_1
$$

其中：

- $X_0$：高斯噪声轨迹。
- $X_1$：真实驾驶轨迹。
- $c$：场景条件。

你已经训练：

$$
b_\theta(x,t,c)
$$

来预测速度。

那么论文给出的启发主要有三个。

### 第一，Flow Matching 可以扩展为随机采样，但需要 Score

确定性：

$$
dX_t=b_\theta(X_t,t,c)\,dt
$$

随机：

$$
\boxed{
dX_t=
[b_\theta(X_t,t,c)+\varepsilon(t)s_\theta(X_t,t,c)]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
$$

这里：

$$
s_\theta(x,t,c)
\approx
\nabla_x\log\rho_t(x\mid c)
$$

因此，如果你希望使用 SDE 做强化学习探索，就需要考虑如何获得条件 Score。

不过，对于高斯噪声起点和线性 CondOT，**并不一定需要额外训练一个 Score 网络**。

因为可以利用速度场与条件去噪器之间的关系推导 Score。

例如：

$$
X_t=(1-t)Z+tX_1
$$

有：

$$
b_t(x)
=
\frac{\eta_1(t,x)-x}{1-t}
$$

另一方面，根据高斯条件分布：

$$
s_t(x)
=
-\frac{\eta_z(t,x)}{1-t}
$$

而：

$$
x=(1-t)\eta_z+t\eta_1
$$

因此：

$$
\eta_z=\frac{x-t\eta_1}{1-t}
$$

代入：

$$
s_t(x)
=
-\frac{x-t\eta_1}{(1-t)^2}
$$

再利用：

$$
\eta_1=x+(1-t)b_t
$$

得到：

$$
x-t\eta_1
=
(1-t)x-t(1-t)b_t
$$

所以：

$$
\boxed{
s_t(x)
=
\frac{tb_t(x)-x}{1-t}
}
$$

这意味着：

$$
\boxed{
\text{在线性 CondOT 且源分布为标准高斯时，理论上可以从速度场恢复 Score。}
}
$$

这是一个对你非常有用的结论。

不过在：

$$
t\rightarrow1
$$

附近会出现数值不稳定，因此实际使用时需要处理端点问题。

### 第二，SDE 的探索随机性与最终目标分布不是同一个概念

改变：

$$
\varepsilon(t)
$$

能够改变生成路径的随机程度。

但是在理想条件下，它并不会改变最终：

$$
p(X_1\mid c)
$$

因此：

**如果目标是强化学习探索，增加 SDE 随机性并不意味着理论上扩大了最终目标分布的支持集。**

它主要改变采样路径。

在有限步数、近似网络和奖励引导存在时，实际生成分布可能发生变化，但这已经超出了理想边缘一致性结论。

### 第三，不能只关注速度 MSE

论文的 Likelihood Control 分析说明：

$$
\mathbb E\|\hat b-b\|^2
$$

并不能完整描述 ODE 生成分布的误差。

对于轨迹规划，除了速度回归损失，还应该关注：

- 最终轨迹分布的拟合质量；
- ODE 数值积分误差；
- 不同采样步数下的性能变化；
- 生成轨迹的多模态覆盖；
- 碰撞率、舒适性和规划成功率。

尤其是当你希望将 Flow Matching 作为强化学习策略时，还需要区分最终轨迹概率与完整 SDE 路径概率。

---

# 二十二、整篇论文总结

Stochastic Interpolants 最核心的贡献，可以概括为四个层次。

| 层次 | 核心公式 | 作用 |
|---|---|---|
| 随机插值 | $X_t=I(t,X_0,X_1)+\gamma Z$ | 定义连接两个分布的概率路径 |
| 速度与 Score | $b=\mathbb E[\dot X_t\mid X_t]$，$s=-\mathbb E[Z\mid X_t]/\gamma$ | 构造可学习的动力学 |
| ODE / SDE | $dX=b\,dt$，$dX=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW$ | 统一确定性和随机生成 |
| 理论分析 | KL 控制、Schrödinger Bridge | 研究误差、随机性与最优运输 |

### 我认为最值得记住的五个结论

**1. 随机插值本身不一定是 SDE。**

它首先是一个通过端点样本和潜变量构造的随机过程。

**2. 速度场是条件路径速度的条件期望。**

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
$$

这解释了为什么 Flow Matching 可以使用条件速度进行监督。

**3. 同一条边缘概率路径可以对应一族 ODE 和 SDE。**

$$
\boxed{
dX_t=
[b_t+\varepsilon s_t]dt+
\sqrt{2\varepsilon}dW_t
}
$$

其中 $\varepsilon$ 可以调节随机性。

**4. 训练误差对 ODE 和 SDE 的分布误差影响不同。**

SDE 的扩散项可以帮助建立基于漂移误差的 KL 上界，而 ODE 的对应分析通常需要额外控制分布的 Score 差异。

**5. Stochastic Interpolants 将 Flow Matching、Score-Based Diffusion、Denoising、Rectified Flow 和 Schrödinger Bridge 放到了同一个概率路径框架中。**

---

**最后，与你当前研究最直接相关的，是下面这个关系：**

对于线性 CondOT：

$$
\boxed{
X_t=(1-t)Z+tX_1
}
$$

理论上：

$$
\boxed{
s_t(x)=\frac{tb_t(x)-x}{1-t}
}
$$

因此，如果已经训练了准确的 Flow Matching 速度场，就可以构造：

$$
\boxed{
dX_t=
\left[
b_\theta(X_t,t)
+
\varepsilon(t)
\frac{tb_\theta(X_t,t)-X_t}{1-t}
\right]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
$$

这为你之前研究的 **Flow Matching → SDE 采样 → 强化学习探索** 提供了一条直接的数学路径。

但在真正用于 GRPO 或其他强化学习算法之前，还需要进一步解决端点数值稳定性、离散采样概率密度和策略更新后的分布一致性问题。

**参考论文：** [Stochastic Interpolants: A Unifying Framework for Flows and Diffusions](https://arxiv.org/abs/2303.08797v3)。